# metabeta — introductory demo

This notebook walks through the **amortized inference API** on a real dataset: loading data,
preprocessing, running posterior sampling, and inspecting results — no MCMC, no training required.

Results are shown twice:
- **Part A — standardized scale**: the model's internal representation; useful for reading
  off prior–posterior contrast and for predictive checks.
- **Part B — original scale**: back-transformed estimates in interpretable units.


In [ ]:
from pathlib import Path

import pandas as pd
import torch

from metabeta.models.router import Router
from metabeta.plotting import plotParameters
from metabeta.utils.evaluation import Proposal


## 1. Load the dataset

The `sleepstudy` dataset (Belenky et al. 2003) measures reaction times (ms) for 18 subjects
over 10 days of sleep restriction.  It is a classic two-level dataset: observations nested
within subjects.


In [ ]:
DATASET_PATH = Path('../metabeta/datasets/from-r/parquet/sleep.parquet')

df = pd.read_parquet(DATASET_PATH)
print(f'{len(df)} observations, {df["group"].nunique()} subjects')
df.head()


## 2. Specify a formula

We use a standard lme4-style formula.  `Days` enters as a fixed slope; a random intercept
is estimated per subject.


In [ ]:
FORMULA = 'y ~ Days + (1 | group)'


## 3. Load the model

`Router` loads a joint checkpoint — a single `.pt` file that bundles one or more trained
submodels with their routing metadata.  Submodels are instantiated lazily on first use.


In [ ]:
JOINT_PATH = Path('../metabeta/outputs/checkpoints/joint_normal_v1.pt')

mb = Router(JOINT_PATH, device='cpu')
print('Submodels:', [s['id'] for s in mb.submodels])


## 4. Preprocessing

`prepareData` with `stage='preprocessed'` fits a `DataPreprocessor` on the dataframe and
returns the standardized arrays before any model-specific padding or collation.

The preprocessor z-scores both `y` and the continuous predictors `X` (storing `y_mean`,
`sd_y`, `x_means`, `x_stds` for later back-transformation).  All downstream steps operate
on these standardized values internally.


In [ ]:
prep = mb.prepareData(df, formula=FORMULA, fit_preprocessor=True, stage='preprocessed')

print('Keys     :', list(prep.keys()))
print(f"Shape    : n={prep['n']}, m={prep['m']}, d={prep['d']}")
print(f"y scale  : mean={float(prep['y_mean']):.2f}, sd={float(prep['sd_y']):.2f}")
print(f"Columns  : {prep['columns'].tolist()}")


## 5. Inference

`mb.sample()` accepts the preprocessed dict directly (skipping the DataFrame stage).
It builds the design matrices from `formula`, routes to the smallest compatible submodel,
and draws posterior samples.

Passing `diagnostics=True` additionally computes the posterior predictive, from which
R², LOO-NLL, and Pareto k are derived.


In [ ]:
result = mb.sample(
    prep,
    formula=FORMULA,
    n_samples=1000,
    diagnostics=True,
)

print('Routed to submodel  :', result.routes)
print('ffx   (B, S, d)     :', result.proposal.ffx.shape)
print('rfx   (B, m, S, q)  :', result.proposal.rfx.shape)
print('sigma_rfx (B, S, q) :', result.proposal.sigma_rfx.shape)
print('sigma_eps (B, S)    :', result.proposal.sigma_eps.shape)


---

## Part A — Standardized scale

On this scale the model operates in the fully standardized space (★):
- Fixed-effect slopes are Δy per SD of the predictor.
- The intercept is the expected y (in original y units) at the mean of each predictor.
- σ_rfx and σ_ε are in original y units.

Priors are most naturally expressed here because the model's default priors are defined on
the ★ scale.


### Summary table


In [ ]:
print(mb.posteriorSummary(result))


### Prior vs. posterior

The pair grid shows the joint posterior over global parameters.  Prior marginals (blue) are
overlaid on the diagonal to make the prior–posterior update visible: a narrow posterior
relative to the prior means the data were informative.

Layout: marginal KDEs on the diagonal, scatter in the upper triangle, KDE contours in the
lower triangle.


In [ ]:
def makePriorProposal(result, n_samples: int = 1_000) -> Proposal:
    """Draw prior samples from the metadata stored in a RouterResult."""
    pp = result.prior_params
    sd_y = result.scale_info.y_std if result.scale_info is not None else 1.0
    d = len(result.param_names['ffx'])
    q = len(result.param_names['sigma_rfx'])

    # FFX: Normal(nu, tau) on ★ scale; multiply by sd_y to match rescaled posterior
    tau_f = torch.tensor(pp['tau_ffx'][0, :d], dtype=torch.float32) * sd_y
    nu_f  = torch.tensor(pp['nu_ffx'][0,  :d], dtype=torch.float32) * sd_y
    ffx   = torch.distributions.Normal(nu_f, tau_f).sample((n_samples,))  # (S, d)

    # sigma_rfx: HalfNormal(tau) on ★ scale; multiply by sd_y
    tau_r = torch.tensor(pp['tau_rfx'][0, :q], dtype=torch.float32) * sd_y
    srfx  = torch.distributions.HalfNormal(tau_r).sample((n_samples,))    # (S, q)

    # sigma_eps: default HalfNormal(2.5) on ★ scale
    seps  = torch.distributions.HalfNormal(torch.tensor([2.5 * sd_y])).sample((n_samples,))  # (S, 1)

    samples_g = torch.cat([ffx, srfx, seps], dim=-1).unsqueeze(0)  # (1, S, d+q+1)
    return Proposal(
        proposed={
            'global': {'samples': samples_g, 'log_prob': torch.zeros(1, n_samples)},
            'local':  {'samples': torch.zeros(1, 1, n_samples, q),
                       'log_prob': torch.zeros(1, 1, n_samples)},
        },
        has_sigma_eps=True,
    )


prior = makePriorProposal(result)

param_names = result.param_names or {}
n_ffx  = len(param_names.get('ffx', []))
n_srfx = len(param_names.get('sigma_rfx', []))
names  = (
    param_names.get('ffx', [])
    + [f'σ_{n}' for n in param_names.get('sigma_rfx', [])]
    + (['σ_ε'] if result.proposal.has_sigma_eps else [])
)

plotParameters(
    result.proposal,
    prior=prior,
    names=names,
    d_active=n_ffx,
    q_active=n_srfx,
);


### Posterior predictive density

`plotPPD` overlays the observed y KDE (black) against the 95% envelope of replicated-data
KDEs drawn from the posterior predictive.  y is shown on the standardized scale used
internally by the model — the *shape*, not the axis values, is what matters here.


In [ ]:
mb.plotPPD(result)


---

## Part B — Original scale

`scale_info` stores the preprocessor statistics (`y_mean`, `sd_y`, `x_means`, `x_stds`)
so that every quantity can be expressed in the original units of `y` and `X`:
- Intercept: expected reaction time (ms) at mean `Days`.
- `Days` slope: Δ reaction time (ms) per additional day.
- σ_rfx and σ_ε are already in ms (unchanged from Part A).


### Dataset trajectories

Each subject's reaction time increases over days of sleep restriction.  Between-subject
variability in baseline and growth rate motivates the random-intercept model.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
for _, grp in df.groupby('group'):
    ax.plot(grp['Days'], grp['y'], 'o-', alpha=0.55, lw=1.5, ms=5)
ax.set_xlabel('Days of sleep restriction')
ax.set_ylabel('Reaction time (ms)')
ax.set_title('sleepstudy: per-subject trajectories (N=18)')
ax.grid(alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()


### Summary table


In [ ]:
print(mb.posteriorSummary(result, x_scale='original'))


### Per-group random effects


In [ ]:
print(mb.rfxTable(result))


### Posterior parameters

The same pair grid as Part A, but with fixed-effect samples back-transformed to original
units via `scale_info`.


In [ ]:
# Back-transform FFX samples to original scale; sigma_rfx / sigma_eps are already in ms.
ffx_orig  = result.scale_info.to_original_scale(result.proposal.ffx[0])   # (S, d)
srfx      = result.proposal.sigma_rfx[0]                                  # (S, q)
seps      = result.proposal.sigma_eps[0].unsqueeze(-1)                    # (S, 1)
samples_g = torch.cat([ffx_orig, srfx, seps], dim=-1).unsqueeze(0)        # (1, S, d+q+1)

orig_proposal = Proposal(
    proposed={
        'global': {'samples': samples_g, 'log_prob': result.proposal.log_prob_g},
        'local':  result.proposal.data['local'],
    },
    has_sigma_eps=True,
)

plotParameters(
    orig_proposal,
    names=names,
    d_active=n_ffx,
    q_active=n_srfx,
);
